In [11]:
import mitsuba as mi
import drjit as dr

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
import time
import os
import copy
import math

mi.set_variant('cuda_ad_rgb', 'metal_ad_rgb', 'llvm_ad_rgb', 'scalar_rgb')
print("Mitsuba variant:", mi.variant())
print("PyTorch CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))

Mitsuba variant: cuda_ad_rgb
PyTorch CUDA: True
GPU:  NVIDIA GeForce RTX 3060


In [12]:
class DataPathTracer(mi.SamplingIntegrator):
    def __init__(self, props):
        super().__init__(props)
        self.max_depth = 20
        self.rr_depth = 100

    def mis_weight(self, pdf_a, pdf_b):
        a2 = pdf_a * pdf_a
        b2 = pdf_b * pdf_b

        w = a2 / (a2 + b2)
        return dr.detach(dr.select(dr.isfinite(w), w, 0.0))

    def collect_paths(self, scene, sampler, ray, active=True):
        ctx = mi.BSDFContext()

        ray = mi.Ray3f(ray)
        beta = mi.Spectrum(1.0)
        eta = mi.Float(1.0)

        prev_si = dr.zeros(mi.Interaction3f)
        prev_bsdf_pdf = mi.Float(1.0)
        prev_bsdf_delta = mi.Bool(True)

        records = []
        L_check = mi.Spectrum(0.0)

        for depth in range(self.max_depth):
            # Trace current ray
            active_before = mi.Bool(active)
            si = scene.ray_intersect(ray, active=active_before)
            valid_hit = active_before & si.is_valid()

            # Evaluate surface or environment emitter
            emitter = si.emitter(scene, active=active_before)
            is_emitter = emitter != None

            # Current path state
            dir_in = -ray.d
            beta_before = mi.Spectrum(beta)

            # Emitter contribution from the current ray
            direct_sample = mi.DirectionSample3f(scene, si, prev_si)
            emitter_pdf = scene.pdf_emitter_direction(
                prev_si,
                direct_sample,
                active=active_before & is_emitter & ~prev_bsdf_delta
            )

            mis_bsdf = self.mis_weight(prev_bsdf_pdf, emitter_pdf)
            Le = emitter.eval(si, active=active_before & is_emitter)
            emitter_hit = Le * mis_bsdf

            # Continue only from a valid surface
            active_next = valid_hit & (depth + 1 < self.max_depth)
            bsdf = si.bsdf()

            albedo = bsdf.eval_diffuse_reflectance(si, active=valid_hit)

            # Next-event estimation
            active_emitter = (
                active_next &
                mi.has_flag(bsdf.flags(), mi.BSDFFlags.Smooth)
            )

            ds, emitter_weight = scene.sample_emitter_direction(
                si,
                sampler.next_2d(active_emitter),
                test_visibility=True,
                active=active_emitter
            )

            active_emitter &= ds.pdf > 0
            wo_emitter = si.to_local(ds.d)

            bsdf_value, bsdf_pdf = bsdf.eval_pdf(
                ctx,
                si,
                wo_emitter,
                active_emitter
            )

            mis_emitter = dr.select(
                ds.delta,
                1.0,
                self.mis_weight(ds.pdf, bsdf_pdf)
            )

            nee = dr.select(
                active_emitter,
                bsdf_value * emitter_weight * mis_emitter,
                0.0
            )

            # Contribution generated at this bounce
            bounce_estimator = emitter_hit + nee
            path_contribution = beta_before * bounce_estimator
            L_check += path_contribution

            # Sample the next BSDF direction
            bs, bsdf_weight = bsdf.sample(
                ctx,
                si,
                sampler.next_1d(),
                sampler.next_2d(),
                active_next
            )

            valid_bsdf = active_next & (bs.pdf > 0)
            dir_out = si.to_world(bs.wo)
            next_ray = si.spawn_ray(dir_out)

            # Update transport state
            beta *= bsdf_weight
            eta *= bs.eta

            # Russian roulette
            next_depth = depth + 1
            throughput_max = dr.max(mi.unpolarized_spectrum(beta))

            rr_probability = dr.minimum(
                throughput_max * dr.square(eta),
                0.95
            )

            rr_active = (
                valid_bsdf &
                (throughput_max != 0.0) &
                (next_depth >= self.rr_depth)
            )

            rr_continue = (
                sampler.next_1d(rr_active) < rr_probability
            )

            beta[rr_active] *= dr.rcp(
                dr.detach(rr_probability)
            )

            # Rays that actually continue after Russian roulette
            valid_next = (
                valid_bsdf &
                (throughput_max != 0.0) &
                (~rr_active | rr_continue)
            )

            # Save current bounce
            records.append({
                "depth": depth,

                # Path validity
                "active": mi.Bool(active_before),
                "valid_hit": mi.Bool(valid_hit),
                "is_emitter": mi.Bool(is_emitter),
                "valid_bsdf": mi.Bool(valid_bsdf),
                "valid_next": mi.Bool(valid_next),

                # Geometry
                "position": mi.Point3f(si.p),
                "normal": mi.Normal3f(si.sh_frame.n),

                # Directions
                "dir_in": mi.Vector3f(dir_in),
                "dir_out": mi.Vector3f(dir_out),

                # Transport state
                "beta": mi.Spectrum(beta_before),
                "bsdf_weight": mi.Spectrum(bsdf_weight),
                "bsdf_pdf": mi.Float(bs.pdf),

                # Light estimator
                "emitter_hit": mi.Spectrum(emitter_hit),
                "nee": mi.Spectrum(nee),
                "bounce_estimator": mi.Spectrum(bounce_estimator),
                "path_contribution": mi.Spectrum(path_contribution),

                # Russian roulette
                "rr_active": mi.Bool(rr_active),
                "rr_probability": mi.Float(rr_probability),

                # Neural Training Features
                "albedo": mi.Spectrum(albedo),
                "prev_bsdf_pdf": mi.Float(prev_bsdf_pdf),
                "prev_bsdf_delta": mi.Bool(prev_bsdf_delta),
            })

            # Save information needed for MIS at the next vertex
            prev_si = mi.Interaction3f(si)
            prev_bsdf_pdf = bs.pdf
            prev_bsdf_delta = mi.has_flag(
                bs.sampled_type,
                mi.BSDFFlags.Delta
            )

            # Move to the next bounce
            ray = next_ray
            active = valid_next

        return records, L_check

    @dr.syntax
    def sample(self, scene, sampler, ray, medium=None, active=True):
        # init 
        ctx = mi.BSDFContext()
        ray = mi.Ray3f(ray)
        L = mi.Spectrum(0.0)
        beta = mi.Spectrum(1.0)

        eta = mi.Float(1.0)
        depth = mi.UInt32(0)

        valid_ray = mi.Bool(False)

        # Information from the previous vertex.
        prev_si = dr.zeros(mi.Interaction3f)
        prev_bsdf_pdf = mi.Float(1.0)
        prev_bsdf_delta = mi.Bool(True)

        # main loop
        while active:
            si = scene.ray_intersect(ray, active=active)
            emitter = si.emitter(scene, active=active)
            is_emitter = emitter != None

            # Probability that NEE could have generated 
            direct_sample = mi.DirectionSample3f(scene, si, prev_si)
            emitter_pdf = scene.pdf_emitter_direction(
                prev_si, 
                direct_sample, 
                active=active & is_emitter & ~prev_bsdf_delta
            )

            mis_bsdf = self.mis_weight(prev_bsdf_pdf, emitter_pdf)

            Le = emitter.eval(si, active=active & is_emitter)

            L += beta * Le * mis_bsdf
            valid_ray |= active & is_emitter

            is_hit = si.is_valid()
            active_next = (active & is_hit & (depth + 1 < self.max_depth))

            bsdf = si.bsdf()
            active_emitter = (active_next & mi.has_flag(bsdf.flags(), mi.BSDFFlags.Smooth))
            ds, emitter_weight = scene.sample_emitter_direction(
                si, 
                sampler.next_2d(active_emitter),
                test_visibility=True,
                active=active_emitter
            )

            active_emitter &= ds.pdf > 0
            wo_emitter = si.to_local(ds.d)

            bsdf_value, bsdf_pdf = bsdf.eval_pdf(
                ctx, 
                si,
                wo_emitter,
                active_emitter
            )

            mis_emitter = dr.select(ds.delta, 1.0, self.mis_weight(ds.pdf, bsdf_pdf))

            direct_contribution = (beta * bsdf_value * emitter_weight * mis_emitter)
            L += dr.select(active_emitter, direct_contribution, 0.0)

            bs, bsdf_weight = bsdf.sample(ctx, si, sampler.next_1d(), sampler.next_2d(), active_next)
            active_bsdf = active_next & (bs.pdf > 0)

            new_direction = si.to_world(bs.wo)
            ray = si.spawn_ray(new_direction)

            beta *= bsdf_weight
            eta *= bs.eta

            non_null = (active_bsdf & ~mi.has_flag(bs.sampled_type, mi.BSDFFlags.Null))
            valid_ray |= non_null

            prev_si = mi.Interaction3f(si)
            prev_bsdf_pdf = bs.pdf
            prev_bsdf_delta = mi.has_flag(bs.sampled_type, mi.BSDFFlags.Delta)

            depth += 1

            throughput_max = dr.max(mi.unpolarized_spectrum(beta))

            rr_probability = dr.minimum(
                throughput_max * dr.square(eta),
                0.95
            )

            rr_active = (
                active_bsdf &
                (throughput_max != 0.0) &
                (depth >= self.rr_depth)
            )

            rr_continue = (
                sampler.next_1d(rr_active) < rr_probability
            )

            beta[rr_active] *= dr.rcp(
                dr.detach(rr_probability)
            )

            active = (
                active_bsdf &
                (throughput_max != 0.0) &
                (~rr_active | rr_continue)
            )

        return dr.select(valid_ray, L, 0.0), valid_ray, []

In [13]:
SCENE_PATH = "assets/cornell-box/scene.xml"
NUM_SAMPLES = 4096
SEED = 42

scene = mi.load_file(SCENE_PATH)
sensor = scene.sensors()[0]

print("Scene       :", SCENE_PATH)
print("Num samples :", NUM_SAMPLES)
print("Seed        :", SEED)

Scene       : assets/cornell-box/scene.xml
Num samples : 4096
Seed        : 42


In [14]:

# -----------------------------------------------
# Cell 1: Validate Path Data Collection
# -----------------------------------------------

# 1. Camera ray sampling
camera_sampler = mi.load_dict({
    "type": "independent"
})
camera_sampler.seed(SEED, NUM_SAMPLES)

ray, _ = sensor.sample_ray(
    time=mi.Float(0.0),
    sample1=camera_sampler.next_1d(),
    sample2=camera_sampler.next_2d(),
    sample3=camera_sampler.next_2d()
)

active_mask = dr.full(
    mi.Bool, True, NUM_SAMPLES
)

# 2. Create independent samplers with same seed
sampler_collect = mi.load_dict({
    "type": "independent"
})
sampler_reference = mi.load_dict({
    "type": "independent"
})

sampler_collect.seed(SEED + 1, NUM_SAMPLES)
sampler_reference.seed(SEED + 1, NUM_SAMPLES)

# 3. Initialize Path Tracer
tracer = DataPathTracer(mi.Properties())

# 4. Collect bounce-wise path records
records, L_check = tracer.collect_paths(
    scene,
    sampler_collect,
    ray,
    active=active_mask
)

# 5. Compute reference Path Tracing result
L_reference, valid_reference, _ = tracer.sample(
    scene,
    sampler_reference,
    ray,
    active=active_mask
)

# 6. Compare estimators
L_check_masked = dr.select(
    valid_reference,
    L_check,
    0.0
)

error = dr.abs(L_reference - L_check_masked)

mean_error = float(
    dr.mean(mi.luminance(error))[0]
)
max_error = float(
    dr.max(mi.luminance(error))[0]
)

print("Estimator Validation")
print("-------------------------------")
print("Mean Error :", mean_error)
print("Max Error  :", max_error)

print("Recorded features:")
print(list(records[0].keys()))

assert "albedo" in records[0]
assert "prev_bsdf_pdf" in records[0]
assert "prev_bsdf_delta" in records[0]

print("\nAll training features: OK")

# 7. Verify neural training features
required_features = [
    "position",
    "normal",
    "dir_in",
    "albedo",
    "prev_bsdf_pdf",
    "prev_bsdf_delta",
    "bsdf_weight",
    "bounce_estimator",
    "valid_hit",
    "valid_next"
]

for feature in required_features:
    assert feature in records[0], (
        f"Missing feature: {feature}"
    )

print("\nTraining Features: OK")

# 8. Analyze valid paths for each bounce
print("\nBounce Statistics")
print("-------------------------------")

for record in records:
    depth = record["depth"]
    valid_count = int(
        dr.count(record["valid_hit"])[0]
    )

    print(
        f"Depth {depth:2d} | "
        f"Valid paths: {valid_count}"
    )

# 9. Final validation
assert mean_error < 1e-5
assert max_error < 1e-4

print("\nPath Collection Validation: PASSED")


Estimator Validation
-------------------------------
Mean Error : 1.2731753429306991e-09
Max Error  : 5.6378439694526605e-08
Recorded features:
['depth', 'active', 'valid_hit', 'is_emitter', 'valid_bsdf', 'valid_next', 'position', 'normal', 'dir_in', 'dir_out', 'beta', 'bsdf_weight', 'bsdf_pdf', 'emitter_hit', 'nee', 'bounce_estimator', 'path_contribution', 'rr_active', 'rr_probability', 'albedo', 'prev_bsdf_pdf', 'prev_bsdf_delta']

All training features: OK

Training Features: OK

Bounce Statistics
-------------------------------
Depth  0 | Valid paths: 4096
Depth  1 | Valid paths: 3055
Depth  2 | Valid paths: 2594
Depth  3 | Valid paths: 2183
Depth  4 | Valid paths: 1889
Depth  5 | Valid paths: 1637
Depth  6 | Valid paths: 1440
Depth  7 | Valid paths: 1233
Depth  8 | Valid paths: 1084
Depth  9 | Valid paths: 955
Depth 10 | Valid paths: 853
Depth 11 | Valid paths: 758
Depth 12 | Valid paths: 675
Depth 13 | Valid paths: 593
Depth 14 | Valid paths: 521
Depth 15 | Valid paths: 462
Depth

In [15]:

# --------------------------------------------------
# Cell 2: Generate 64-spp Path Dataset
# --------------------------------------------------

import numpy as np

SPP = 64
NUM_PIXELS = 64
DATA_SEED = 100

# 1. Get film resolution
film = sensor.film()
width = int(film.crop_size()[0])
height = int(film.crop_size()[1])

print("Film resolution:", width, "x", height)

# 2. Select random pixels
rng = np.random.default_rng(DATA_SEED)

pixel_ids = rng.choice(
    width * height,
    size=NUM_PIXELS,
    replace=False
)

# 3. Generate exactly 64 samples per pixel
pixel_ids_per_ray = np.repeat(pixel_ids, SPP)

num_rays = len(pixel_ids_per_ray)

px = pixel_ids_per_ray % width
py = pixel_ids_per_ray // width

# Subpixel jitter
jitter_x = rng.random(num_rays).astype(np.float32)
jitter_y = rng.random(num_rays).astype(np.float32)

# Normalized film coordinates
u = (px + jitter_x) / width
v = (py + jitter_y) / height

# 4. Generate camera rays
camera_sampler = mi.load_dict({
    "type": "independent"
})
camera_sampler.seed(DATA_SEED + 1, num_rays)

ray_spp, camera_weight = sensor.sample_ray(
    time=mi.Float(0.0),
    sample1=camera_sampler.next_1d(),
    sample2=mi.Point2f(
        mi.Float(u.astype(np.float32)),
        mi.Float(v.astype(np.float32))
    ),
    sample3=camera_sampler.next_2d()
)

# 5. Trace all camera rays
path_sampler = mi.load_dict({
    "type": "independent"
})
path_sampler.seed(DATA_SEED + 2, num_rays)

records_spp, L_spp = tracer.collect_paths(
    scene,
    path_sampler,
    ray_spp,
    active=dr.full(mi.Bool, True, num_rays)
)

# 6. Validate collected data
assert num_rays == NUM_PIXELS * SPP
assert len(records_spp) == tracer.max_depth

print("\n64-spp Dataset")
print("---------------------------")
print("Selected pixels :", NUM_PIXELS)
print("SPP             :", SPP)
print("Total rays      :", num_rays)
print("Bounce records  :", len(records_spp))

print("\nValid paths per bounce")
for record in records_spp:
    count = int(dr.count(record["valid_hit"])[0])
    print(f"Depth {record['depth']:2d}: {count}")

# 7. Compute mean luminance per selected pixel
ray_luminance = np.asarray(
    mi.luminance(L_spp),
    dtype=np.float32
)

assert np.isfinite(ray_luminance).all()

pixel_mean = ray_luminance.reshape(
    NUM_PIXELS, SPP
).mean(axis=1)

print("\nMean luminance (64 spp)")
print("---------------------------")
print("Mean :", pixel_mean.mean())
print("Min  :", pixel_mean.min())
print("Max  :", pixel_mean.max())

print("\n64-spp Path Collection: PASSED")


Film resolution: 1024 x 1024

64-spp Dataset
---------------------------
Selected pixels : 64
SPP             : 64
Total rays      : 4096
Bounce records  : 20

Valid paths per bounce
Depth  0: 4096
Depth  1: 2985
Depth  2: 2469
Depth  3: 2084
Depth  4: 1770
Depth  5: 1540
Depth  6: 1314
Depth  7: 1141
Depth  8: 999
Depth  9: 866
Depth 10: 750
Depth 11: 662
Depth 12: 580
Depth 13: 513
Depth 14: 453
Depth 15: 398
Depth 16: 359
Depth 17: 321
Depth 18: 282
Depth 19: 261

Mean luminance (64 spp)
---------------------------
Mean : 0.071482405
Min  : 0.0041834814
Max  : 0.21848218

64-spp Path Collection: PASSED


In [16]:

# --------------------------------------------------
# Cell 3: Generate Continuation Radiance Targets
# --------------------------------------------------

import numpy as np

num_depths = len(records_spp)

continuation_targets = [None] * num_depths

# Start from the end of each path
R_next = mi.Spectrum(0.0)

# 1. Reconstruct radiance backward
for depth in reversed(range(num_depths)):
    record = records_spp[depth]

    C = record["bounce_estimator"]
    W = record["bsdf_weight"]

    valid_next = record["valid_next"]
    active = record["active"]

    # Contribution from future bounces
    future_radiance = dr.select(
        valid_next,
        W * R_next,
        0.0
    )

    # Continuation radiance at the current bounce
    R = C + future_radiance

    # Preserve environment emitter hits on ray misses
    R = dr.select(active, R, 0.0)

    continuation_targets[depth] = mi.Spectrum(R)
    R_next = R

print("Continuation Targets: Generated")

# 2. Validate reconstruction against Path Tracer
reconstructed = continuation_targets[0]

error = dr.abs(reconstructed - L_spp)

mean_error = float(
    dr.mean(mi.luminance(error))[0]
)

max_error = float(
    dr.max(mi.luminance(error))[0]
)

print("\nTarget Reconstruction Validation")
print("--------------------------------")
print("Mean Error:", mean_error)
print("Max Error :", max_error)

assert mean_error < 1e-5
assert max_error < 1e-4

print("Reconstruction: PASSED")

# 3. Analyze targets by bounce depth
print("\nContinuation Radiance Statistics")
print("--------------------------------")

for depth, record in enumerate(records_spp):
    valid = (
        record["valid_hit"] &
        ~record["is_emitter"]
    )

    num_valid = int(dr.count(valid)[0])

    if num_valid == 0:
        continue

    R = continuation_targets[depth]

    values = np.asarray(
        mi.luminance(R), dtype=np.float32
    )

    mask = np.asarray(valid, dtype=bool)

    valid_values = values[mask]

    print(
        f"Depth {depth:2d} | "
        f"N {num_valid:4d} | "
        f"Mean {valid_values.mean():.6f} | "
        f"Std {valid_values.std():.6f}"
    )

print("\nCell 3: PASSED")


Continuation Targets: Generated

Target Reconstruction Validation
--------------------------------
Mean Error: 2.3067350252148344e-09
Max Error : 6.394029128387047e-08
Reconstruction: PASSED

Continuation Radiance Statistics
--------------------------------
Depth  0 | N 4096 | Mean 0.071482 | Std 0.074040
Depth  1 | N 2969 | Mean 0.074515 | Std 0.089643
Depth  2 | N 2442 | Mean 0.083098 | Std 0.093721
Depth  3 | N 2063 | Mean 0.082971 | Std 0.097095
Depth  4 | N 1762 | Mean 0.081245 | Std 0.089214
Depth  5 | N 1527 | Mean 0.083173 | Std 0.092335
Depth  6 | N 1300 | Mean 0.079550 | Std 0.089676
Depth  7 | N 1132 | Mean 0.077442 | Std 0.090620
Depth  8 | N  993 | Mean 0.078387 | Std 0.087924
Depth  9 | N  860 | Mean 0.080559 | Std 0.090808
Depth 10 | N  747 | Mean 0.083266 | Std 0.086840
Depth 11 | N  658 | Mean 0.079926 | Std 0.090591
Depth 12 | N  572 | Mean 0.075547 | Std 0.082461
Depth 13 | N  510 | Mean 0.077580 | Std 0.088237
Depth 14 | N  451 | Mean 0.078385 | Std 0.088838
Depth 1

In [18]:

# --------------------------------------------------
# Cell 4: Validate Neural Training Targets
# --------------------------------------------------

import numpy as np
from tqdm import tqdm

statistics = []

for depth, record in enumerate(
    tqdm(records_spp, desc="Validating targets", unit="bounce")
):
    valid_hit = np.asarray(
        record["valid_hit"], dtype=bool
    ).reshape(-1)

    is_emitter = np.asarray(
        record["is_emitter"], dtype=bool
    ).reshape(-1)

    # Verify mask dimensions
    assert len(valid_hit) == num_rays
    assert len(is_emitter) == num_rays, (
        f"Depth {depth}: emitter mask shape mismatch "
        f"({len(is_emitter)} vs {num_rays})"
    )

    # Only non-emissive surface vertices
    valid = valid_hit & ~is_emitter

    R = np.asarray(
        mi.luminance(continuation_targets[depth]),
        dtype=np.float32
    ).reshape(-1)

    assert len(R) == num_rays

    values = R[valid]

    assert np.isfinite(values).all(), (
        f"Non-finite target at depth {depth}"
    )

    statistics.append({
        "depth": depth,
        "hits": int(valid_hit.sum()),
        "emitters": int((valid_hit & is_emitter).sum()),
        "training": int(valid.sum()),
        "mean": float(values.mean()) if len(values) else 0.0,
        "std": float(values.std()) if len(values) else 0.0
    })

print("\nNeural Training Target Statistics")
print("-" * 72)

for s in statistics:
    print(
        f"Depth {s['depth']:2d} | "
        f"Hits {s['hits']:4d} | "
        f"Emitters {s['emitters']:4d} | "
        f"Train {s['training']:4d} | "
        f"Mean {s['mean']:.5f} | "
        f"Std {s['std']:.5f}"
    )

total_vertices = sum(s["training"] for s in statistics)

print("\nTotal Training Vertices:", total_vertices)

assert total_vertices > 0, (
    "No non-emissive training vertices. "
    "Check is_emitter mask."
)

print("\nCell 4: PASSED")


Validating targets: 100%|██████████| 20/20 [02:54<00:00,  8.72s/bounce]


Neural Training Target Statistics
------------------------------------------------------------------------
Depth  0 | Hits 4096 | Emitters    0 | Train 4096 | Mean 0.07148 | Std 0.07404
Depth  1 | Hits 2985 | Emitters   16 | Train 2969 | Mean 0.07451 | Std 0.08964
Depth  2 | Hits 2469 | Emitters   27 | Train 2442 | Mean 0.08310 | Std 0.09372
Depth  3 | Hits 2084 | Emitters   21 | Train 2063 | Mean 0.08297 | Std 0.09710
Depth  4 | Hits 1770 | Emitters    8 | Train 1762 | Mean 0.08125 | Std 0.08921
Depth  5 | Hits 1540 | Emitters   13 | Train 1527 | Mean 0.08317 | Std 0.09233
Depth  6 | Hits 1314 | Emitters   14 | Train 1300 | Mean 0.07955 | Std 0.08968
Depth  7 | Hits 1141 | Emitters    9 | Train 1132 | Mean 0.07744 | Std 0.09062
Depth  8 | Hits  999 | Emitters    6 | Train  993 | Mean 0.07839 | Std 0.08792
Depth  9 | Hits  866 | Emitters    6 | Train  860 | Mean 0.08056 | Std 0.09081
Depth 10 | Hits  750 | Emitters    3 | Train  747 | Mean 0.08327 | Std 0.08684
Depth 11 | Hits  662 | 

In [21]:

# --------------------------------------------------
# Cell 5: Convert Path Records to PyTorch Dataset
# --------------------------------------------------

import numpy as np
import torch
from tqdm import tqdm

NUM_DEPTHS = len(records_spp)
NUM_RAYS = len(pixel_ids_per_ray)
NUM_FEATURES = 15

assert NUM_RAYS == NUM_PIXELS * SPP
assert NUM_DEPTHS == tracer.max_depth

# --------------------------------------------------
# 1. Allocate dataset
# --------------------------------------------------

X = np.zeros(
    (NUM_RAYS, NUM_DEPTHS, NUM_FEATURES),
    dtype=np.float32
)

Y = np.zeros(
    (NUM_RAYS, NUM_DEPTHS, 3),
    dtype=np.float32
)

M = np.zeros(
    (NUM_RAYS, NUM_DEPTHS),
    dtype=np.float32
)

# --------------------------------------------------
# 2. Scene bounding box
# --------------------------------------------------

bbox = scene.bbox()

bbox_min = np.array([
    float(bbox.min.x),
    float(bbox.min.y),
    float(bbox.min.z)
], dtype=np.float32)

bbox_max = np.array([
    float(bbox.max.x),
    float(bbox.max.y),
    float(bbox.max.z)
], dtype=np.float32)

bbox_size = np.maximum(
    bbox_max - bbox_min,
    1e-6
)

# --------------------------------------------------
# 3. Conversion functions
# --------------------------------------------------

def to_ray_array(value, name, dtype=np.float32):
    """
    Convert Mitsuba scalar or Dr.Jit array into
    a NumPy array of shape (NUM_RAYS,).

    Scalar values are broadcast to all rays.
    """

    arr = np.asarray(
        value,
        dtype=dtype
    ).reshape(-1)

    if arr.size == 1:
        arr = np.full(
            NUM_RAYS,
            arr.item(),
            dtype=dtype
        )

    elif arr.size != NUM_RAYS:
        raise ValueError(
            f"{name}: expected {NUM_RAYS} elements, "
            f"got {arr.size}, shape={arr.shape}"
        )

    return arr


def to_vec3(v, name="vector"):
    return np.stack([
        to_ray_array(v.x, f"{name}.x"),
        to_ray_array(v.y, f"{name}.y"),
        to_ray_array(v.z, f"{name}.z")
    ], axis=-1)


def to_rgb(v, name="spectrum"):
    return np.stack([
        to_ray_array(v[0], f"{name}.r"),
        to_ray_array(v[1], f"{name}.g"),
        to_ray_array(v[2], f"{name}.b")
    ], axis=-1)


# --------------------------------------------------
# 4. Process each bounce
# --------------------------------------------------

for depth in tqdm(
    range(NUM_DEPTHS),
    desc="Building PyTorch Dataset",
    unit="bounce"
):
    rec = records_spp[depth]

    # ----------------------------------------------
    # Validity masks
    # ----------------------------------------------

    valid_hit = to_ray_array(
        rec["valid_hit"],
        "valid_hit",
        dtype=np.bool_
    )

    is_emitter = to_ray_array(
        rec["is_emitter"],
        "is_emitter",
        dtype=np.bool_
    )

    valid = valid_hit & ~is_emitter

    # Exclude final depth:
    # continuation is forcibly truncated.
    if depth == NUM_DEPTHS - 1:
        valid[:] = False

    # ----------------------------------------------
    # Geometry
    # ----------------------------------------------

    position = to_vec3(
        rec["position"], "position"
    )

    normal = to_vec3(
        rec["normal"], "normal"
    )

    direction = to_vec3(
        rec["dir_in"], "dir_in"
    )

    # Normalize position to [0, 1)
    position = (
        position - bbox_min
    ) / bbox_size

    position = np.clip(
        position,
        0.0,
        1.0 - 1e-6
    )

    # ----------------------------------------------
    # Material
    # ----------------------------------------------

    albedo = to_rgb(
        rec["albedo"], "albedo"
    )

    # ----------------------------------------------
    # Previous sampling information
    # ----------------------------------------------

    prev_pdf = to_ray_array(
        rec["prev_bsdf_pdf"],
        "prev_bsdf_pdf"
    ).reshape(NUM_RAYS, 1)

    prev_pdf = np.log1p(
        np.maximum(prev_pdf, 0.0)
    )

    prev_delta = to_ray_array(
        rec["prev_bsdf_delta"],
        "prev_bsdf_delta"
    ).reshape(NUM_RAYS, 1)

    # ----------------------------------------------
    # Bounce index
    # ----------------------------------------------

    depth_feature = np.full(
        (NUM_RAYS, 1),
        depth / max(NUM_DEPTHS - 1, 1),
        dtype=np.float32
    )

    # ----------------------------------------------
    # Assemble 15 features
    # ----------------------------------------------

    features = np.concatenate([
        position,       # 3
        normal,         # 3
        direction,      # 3
        albedo,         # 3
        prev_pdf,       # 1
        prev_delta,     # 1
        depth_feature   # 1
    ], axis=-1)

    assert features.shape == (
        NUM_RAYS, NUM_FEATURES
    ), (
        f"Depth {depth}: invalid feature shape "
        f"{features.shape}"
    )

    # ----------------------------------------------
    # RGB continuation target
    # ----------------------------------------------

    target = to_rgb(
        continuation_targets[depth],
        f"target_depth_{depth}"
    )

    assert target.shape == (NUM_RAYS, 3)

    # ----------------------------------------------
    # Validate active samples
    # ----------------------------------------------

    assert np.isfinite(features[valid]).all(), (
        f"Invalid features at depth {depth}"
    )

    assert np.isfinite(target[valid]).all(), (
        f"Invalid target at depth {depth}"
    )

    # ----------------------------------------------
    # Store data
    # ----------------------------------------------

    # Masked entries become zero-padding.
    X[:, depth, :] = np.where(
        valid[:, None],
        features,
        0.0
    )

    Y[:, depth, :] = np.where(
        valid[:, None],
        target,
        0.0
    )

    M[:, depth] = valid.astype(np.float32)

# --------------------------------------------------
# 5. Convert NumPy to PyTorch
# --------------------------------------------------

X_tensor = torch.from_numpy(X.copy())
Y_tensor = torch.from_numpy(Y.copy())
M_tensor = torch.from_numpy(M.copy())

print("\nDataset Shapes")
print("--------------------------------")
print("X:", X_tensor.shape)
print("Y:", Y_tensor.shape)
print("M:", M_tensor.shape)

print("\nDataset Statistics")
print("--------------------------------")
print("Total rays:", NUM_RAYS)
print("Total valid vertices:", int(M_tensor.sum()))

valid_targets = Y_tensor[M_tensor.bool()]

assert valid_targets.numel() > 0

print("Target min:", valid_targets.min().item())
print("Target max:", valid_targets.max().item())
print("Target mean:", valid_targets.mean().item())

# --------------------------------------------------
# 6. Train / Validation split by pixel
# --------------------------------------------------

rng_split = np.random.default_rng(42)

pixel_indices = rng_split.permutation(
    NUM_PIXELS
)

n_train_pixels = int(
    NUM_PIXELS * 0.8
)

train_pixels = pixel_indices[:n_train_pixels]
val_pixels = pixel_indices[n_train_pixels:]

assert len(train_pixels) > 0
assert len(val_pixels) > 0

# Keep all 64 samples from one pixel
# in the same dataset split.

def select_pixel_groups(tensor, indices):

    grouped = tensor.reshape(
        NUM_PIXELS,
        SPP,
        *tensor.shape[1:]
    )

    selected = grouped[indices]

    return selected.reshape(
        -1,
        *tensor.shape[1:]
    ).contiguous()


X_train = select_pixel_groups(
    X_tensor, train_pixels
)

Y_train = select_pixel_groups(
    Y_tensor, train_pixels
)

M_train = select_pixel_groups(
    M_tensor, train_pixels
)

X_val = select_pixel_groups(
    X_tensor, val_pixels
)

Y_val = select_pixel_groups(
    Y_tensor, val_pixels
)

M_val = select_pixel_groups(
    M_tensor, val_pixels
)

print("\nTrain / Validation Split")
print("--------------------------------")
print("Train pixels:", len(train_pixels))
print("Val pixels  :", len(val_pixels))
print("Train paths :", len(X_train))
print("Val paths   :", len(X_val))

print("Train vertices:", int(M_train.sum()))
print("Val vertices  :", int(M_val.sum()))

# --------------------------------------------------
# 7. Final validation
# --------------------------------------------------

assert torch.isfinite(X_tensor).all()
assert torch.isfinite(Y_tensor).all()

assert M_train.sum() > 0
assert M_val.sum() > 0

# Final depth excluded
assert M_tensor[:, -1].sum() == 0

# Check split sample counts
assert len(X_train) == len(train_pixels) * SPP
assert len(X_val) == len(val_pixels) * SPP

# Check feature dimension
assert X_train.shape[-1] == NUM_FEATURES

print("\nCell 5: PASSED")


Building PyTorch Dataset:  35%|███▌      | 7/20 [06:43<12:28, 57.59s/bounce]


KeyboardInterrupt: 